# Base model fine-tuning

Fine-tunes the eight transformer encoders compared in the paper on Humicroedit (SemEval-2020 Task 7, Subtask 1) and writes each model's dev and test predictions to `../predictions/`. The stacking notebook (`02_stacking_ensemble.ipynb`) reads those files.

The notebook was developed on a single GPU (Kaggle / Lightning). Fine-tuning all eight large encoders takes several hours, so the predictions used in the paper are already included in the repository.

In [ ]:
!pip install -q transformers datasets scikit-learn pandas numpy sentencepiece protobuf

In [ ]:
import os
import random
import re

import numpy as np
import pandas as pd
import torch
import torch.nn as nn
from datasets import load_dataset
from sklearn.metrics import mean_squared_error
from torch.utils.data import DataLoader, Dataset
from transformers import AutoModel, AutoTokenizer

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("device:", device, "| torch:", torch.__version__)

In [ ]:
SEED = 42
BATCH_SIZE = 16
MAX_LEN = 128
EPOCHS = 5
PATIENCE = 2
PRED_DIR = "../predictions"


def set_seed(seed=SEED):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)


set_seed()
torch.backends.cudnn.deterministic = True
torch.backends.cudnn.benchmark = False

## 1. Data

All three splits come from the Hugging Face copy of Humicroedit and carry the mean funniness grade (`meanGrade`, 0 to 3). The train split is used for fitting, the dev split for early stopping, and the test split only for the final numbers.

The `original` column marks the word that gets replaced as `<word>`. A few rows use `<word/>` instead, so that is normalised first. We then rebuild the edited headline by substituting the edit word into the marked slot.

In [ ]:
dataset = load_dataset("Deehan1866/Humicroedit")

train_df = dataset["train"].to_pandas()
dev_df = dataset["validation"].to_pandas()
test_df = dataset["test"].to_pandas()


def safe_str(x):
    return "" if x is None or pd.isna(x) else str(x)


def build_edited(original, edit):
    return re.sub(r"<[^>]+>", safe_str(edit), safe_str(original))


def strip_tags(original):
    return re.sub(r"<([^>]+)>", r"\1", safe_str(original))


for df in (train_df, dev_df, test_df):
    df["meanGrade"] = df["meanGrade"].astype(float)
    df["original"] = df["original"].apply(lambda x: re.sub(r"<([^/]+)/>", r"<\1>", str(x)))
    df["edited_headline"] = df.apply(lambda r: build_edited(r["original"], r["edit"]), axis=1)
    df["clean_original"] = df["original"].apply(strip_tags)

print(f"train {len(train_df)} | dev {len(dev_df)} | test {len(test_df)}")
train_df[["clean_original", "edit", "edited_headline", "meanGrade"]].head()

## 2. Metrics

The primary metric is RMSE over the whole split. The task also reports RMSE on the union of the N% funniest and N% least funny headlines (N = 10, 20, 30, 40), which shows how well a model separates the extremes.

In [ ]:
def rmse(y_true, y_pred):
    return float(np.sqrt(mean_squared_error(np.asarray(y_true), np.asarray(y_pred))))


def rmse_top_bottom(y_true, y_pred, pct):
    y_true, y_pred = np.asarray(y_true), np.asarray(y_pred)
    n = int(len(y_true) * pct / 100)
    order = np.argsort(y_true)
    idx = np.concatenate([order[:n], order[-n:]])
    return rmse(y_true[idx], y_pred[idx])


def report(y_true, y_pred, title):
    overall = rmse(y_true, y_pred)
    print(f"{title}: RMSE {overall:.4f}")
    print("   " + " | ".join(f"N{n}% {rmse_top_bottom(y_true, y_pred, n):.4f}" for n in (10, 20, 30, 40)))
    return overall

## 3. Dataset

`HumorDataset` supports the four input formats we tried during development. The paper uses `marked`.

| style | encoder input |
|---|---|
| `edited` | edited headline only |
| `concat` | original headline `[SEP]` edit word |
| `both` | original headline `[SEP]` edited headline |
| `marked` | headline with the edit word wrapped in `<edit>...</edit>` `[SEP]` edited headline |

In [ ]:
class HumorDataset(Dataset):
    def __init__(self, df, tokenizer, max_len=MAX_LEN, input_style="marked"):
        self.df = df.reset_index(drop=True)
        self.tokenizer = tokenizer
        self.max_len = max_len
        self.input_style = input_style

    def __len__(self):
        return len(self.df)

    def _build_input(self, row):
        original, edit, edited = row["clean_original"], row["edit"], row["edited_headline"]
        marked = re.sub(r"<[^>]+>", f"<edit>{edit}</edit>", row["original"])
        if self.input_style == "concat":
            return original, edit
        if self.input_style == "edited":
            return edited, None
        if self.input_style == "both":
            return original, edited
        return marked, edited

    def __getitem__(self, idx):
        row = self.df.iloc[idx]
        text_a, text_b = self._build_input(row)
        enc = self.tokenizer(text_a, text_b, max_length=self.max_len,
                             padding="max_length", truncation=True, return_tensors="pt")
        item = {k: v.squeeze(0) for k, v in enc.items()}
        item["labels"] = torch.tensor(row["meanGrade"], dtype=torch.float)
        return item

## 4. Model and training loop

Each encoder is followed by the same regression head: dropout, a 256-unit linear layer with GELU, more dropout, and a single output. The [CLS] hidden state is used as the sequence representation for every model, including the sentence-embedding checkpoints. The encoder and head are trained jointly with MSE loss, AdamW, cosine annealing, gradient clipping at 1.0 and early stopping on dev RMSE.

In [ ]:
class HumorRegressor(nn.Module):
    def __init__(self, model_name, dropout=0.3):
        super().__init__()
        self.encoder = AutoModel.from_pretrained(model_name)
        hidden = self.encoder.config.hidden_size
        self.regressor = nn.Sequential(
            nn.Dropout(dropout),
            nn.Linear(hidden, 256),
            nn.GELU(),
            nn.Dropout(0.1),
            nn.Linear(256, 1),
        )

    def forward(self, input_ids, attention_mask, token_type_ids=None, labels=None):
        kwargs = dict(input_ids=input_ids, attention_mask=attention_mask)
        if token_type_ids is not None:
            kwargs["token_type_ids"] = token_type_ids
        cls = self.encoder(**kwargs).last_hidden_state[:, 0, :]
        preds = self.regressor(cls).squeeze(-1)
        if labels is None:
            return preds
        return nn.MSELoss()(preds, labels), preds


def train_model(model, train_loader, dev_loader, epochs=EPOCHS, lr=2e-5, patience=PATIENCE):
    optimizer = torch.optim.AdamW(model.parameters(), lr=lr, weight_decay=0.01)
    scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=epochs)
    model.to(device)

    best_rmse, best_state, bad_epochs = float("inf"), None, 0

    for epoch in range(epochs):
        model.train()
        total_loss = 0.0
        for batch in train_loader:
            batch = {k: v.to(device) for k, v in batch.items()}
            labels = batch.pop("labels")
            loss, _ = model(**batch, labels=labels)
            optimizer.zero_grad()
            loss.backward()
            nn.utils.clip_grad_norm_(model.parameters(), 1.0)
            optimizer.step()
            total_loss += loss.item()
        scheduler.step()

        model.eval()
        preds, trues = [], []
        with torch.no_grad():
            for batch in dev_loader:
                batch = {k: v.to(device) for k, v in batch.items()}
                labels = batch.pop("labels")
                _, out = model(**batch, labels=labels)
                preds.extend(out.cpu().numpy())
                trues.extend(labels.cpu().numpy())

        dev_rmse = rmse(trues, preds)
        print(f"  epoch {epoch + 1}/{epochs} | train loss {total_loss / len(train_loader):.4f} | dev RMSE {dev_rmse:.4f}")

        if dev_rmse < best_rmse:
            best_rmse = dev_rmse
            best_state = {k: v.clone() for k, v in model.state_dict().items()}
            bad_epochs = 0
        else:
            bad_epochs += 1
            if bad_epochs >= patience:
                print(f"  early stopping at epoch {epoch + 1}")
                break

    model.load_state_dict(best_state)
    return model


def predict(model, loader):
    model.eval()
    preds = []
    with torch.no_grad():
        for batch in loader:
            batch = {k: v.to(device) for k, v in batch.items() if k != "labels"}
            preds.extend(model(**batch).cpu().numpy())
    return np.clip(np.array(preds), 0.0, 3.0)  # labels live in [0, 3]


def seed_worker(worker_id):
    np.random.seed(SEED + worker_id)
    random.seed(SEED + worker_id)


generator = torch.Generator()
generator.manual_seed(SEED)


def make_loader(df, tokenizer, shuffle):
    return DataLoader(HumorDataset(df, tokenizer), batch_size=BATCH_SIZE, shuffle=shuffle,
                      num_workers=2, worker_init_fn=seed_worker, generator=generator)

## 5. Training one model

`run_model` fine-tunes one checkpoint, scores it on dev and test, and saves the two prediction files that the stacking step uses.

In [ ]:
results = {}


def save_predictions(tag, dev_preds, test_preds):
    os.makedirs(PRED_DIR, exist_ok=True)
    for split, df, preds in (("dev", dev_df, dev_preds), ("test", test_df, test_preds)):
        out = pd.DataFrame({"id": df.index, "label": df["meanGrade"], "prediction": preds})
        out.to_csv(f"{PRED_DIR}/{tag}_{split}.csv", index=False)


def run_model(tag, model_name, lr):
    print(f"\n=== {tag} ({model_name}) ===")
    tokenizer = AutoTokenizer.from_pretrained(model_name)
    tokenizer.add_special_tokens({"additional_special_tokens": ["<edit>", "</edit>"]})

    train_loader = make_loader(train_df, tokenizer, shuffle=True)
    dev_loader = make_loader(dev_df, tokenizer, shuffle=False)
    test_loader = make_loader(test_df, tokenizer, shuffle=False)

    model = HumorRegressor(model_name)
    model.encoder.resize_token_embeddings(len(tokenizer))
    model = train_model(model, train_loader, dev_loader, lr=lr)

    dev_preds = predict(model, dev_loader)
    test_preds = predict(model, test_loader)
    results[tag] = {
        "dev_rmse": report(dev_df["meanGrade"], dev_preds, f"{tag} dev"),
        "test_rmse": report(test_df["meanGrade"], test_preds, f"{tag} test"),
    }
    save_predictions(tag, dev_preds, test_preds)

    del model
    torch.cuda.empty_cache()

## 6. The eight candidates

XLNet and DeBERTa-v3-large are trained with a lower learning rate (1e-5) because they were less stable at 2e-5. Everything else uses 2e-5. Only four of these models enter the final ensemble (DeBERTa, StructBERT, Sentence-BERT and RoBERTa); the other four are kept for the combination analysis in the paper.

In [ ]:
MODELS = [
    # tag,          checkpoint,                                             learning rate
    ("bert",         "bert-large-uncased",                                   2e-5),
    ("roberta",      "roberta-large",                                        2e-5),
    ("structbert",   "bayartsogt/structbert-large",                          2e-5),
    ("mpnet",        "sentence-transformers/nli-mpnet-base-v2",              2e-5),
    ("xlnet",        "xlnet-large-cased",                                    1e-5),
    ("phrasebert",   "whaleloops/phrase-bert",                               2e-5),
    ("sentencebert", "sentence-transformers/bert-base-nli-stsb-mean-tokens", 2e-5),
    ("deberta",      "sileod/deberta-v3-large-tasksource-nli",               1e-5),
]

for tag, checkpoint, lr in MODELS:
    run_model(tag, checkpoint, lr)

In [ ]:
summary = pd.DataFrame(results).T.round(4).sort_values("test_rmse")
summary